# EDA 02: evidencia cuantitativa para posibles sesgos

Este notebook analiza distribución y representación en el dataset **original**. Las diferencias observadas no prueban discriminación: se registran como evidencia para revisión posterior.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'data' / 'raw').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.bias_analysis import distribution, target_by_group, age_groups, findings_matrix
from src.plots import save_count_plot, save_target_group_plot

TABLES = ROOT / 'outputs' / 'tables'
FIGURES = ROOT / 'outputs' / 'figures'
TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(ROOT / 'data' / 'raw' / 'heart.csv')
print('Dataset original cargado:', df.shape)

Dataset original cargado: (1025, 14)


## A. Distribución de target

Pregunta: ¿cómo se distribuyen las clases de la variable objetivo en la muestra?

In [2]:
target_dist = distribution(df, 'target')
display(target_dist)
target_dist.to_csv(TABLES / 'distribucion_target.csv', index=False)
save_count_plot(df['target'].value_counts().sort_index(), 'Distribución de target', 'target', FIGURES / 'distribucion_target.png')

,target,conteo,porcentaje
0,0,499,48.68
1,1,526,51.32


## B. Distribución por sex

Pregunta: ¿los grupos codificados en `sex` tienen una representación similar y cómo varía target dentro de cada grupo?

In [3]:
sex_dist = distribution(df, 'sex')
sex_target = target_by_group(df, 'sex')
display(sex_dist)
display(sex_target)
sex_dist.to_csv(TABLES / 'distribucion_sex.csv', index=False)
sex_target.to_csv(TABLES / 'target_por_sex.csv', index=False)
save_count_plot(df['sex'].value_counts().sort_index(), 'Distribución por sex', 'sex', FIGURES / 'distribucion_sexo.png')
save_target_group_plot(pd.crosstab(df['sex'], df['target']), 'Target por sex', 'sex', FIGURES / 'target_por_sexo.png')

,sex,conteo,porcentaje
0,0,312,30.44
1,1,713,69.56


target,sex,0,1,target_0_porcentaje_grupo,target_1_porcentaje_grupo
0,0,86,226,27.56,72.44
1,1,413,300,57.92,42.08


## C. Distribución por edad

Pregunta: ¿qué rangos etarios están representados y cómo se distribuye target dentro de cada rango? `age_group` se crea solo para análisis y no se guarda en el dataset original.

In [4]:
display(df['age'].describe())
age_group = age_groups(df)
age_dist = distribution(pd.DataFrame({'age_group': age_group}), 'age_group')
age_target = target_by_group(pd.DataFrame({'age_group': age_group, 'target': df['target']}), 'age_group')
display(age_dist)
display(age_target)
age_dist.to_csv(TABLES / 'distribucion_rangos_edad.csv', index=False)
age_target.to_csv(TABLES / 'target_por_rango_edad.csv', index=False)
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.hist(df['age'], bins=12, color='#1f4e79', edgecolor='white')
ax.set_title('Distribución de edad', fontweight='bold')
ax.set_xlabel('Edad')
ax.set_ylabel('Cantidad de registros')
fig.tight_layout()
fig.savefig(FIGURES / 'distribucion_edad.png', dpi=220, bbox_inches='tight')
plt.close(fig)
save_target_group_plot(pd.crosstab(age_group, df['target']), 'Target por rango de edad', 'Rango de edad', FIGURES / 'target_por_edad.png')

count    1025.000000
mean       54.434146
std         9.072290
min        29.000000
25%        48.000000
50%        56.000000
75%        61.000000
max        77.000000
Name: age, dtype: float64

,age_group,conteo,porcentaje
0,<40,57,5.56
1,40-49,237,23.12
2,50-59,422,41.17
3,60-69,275,26.83
4,70+,34,3.32


target,age_group,0,1,target_0_porcentaje_grupo,target_1_porcentaje_grupo
0,<40,15,42,26.32,73.68
1,40-49,80,157,33.76,66.24
2,50-59,216,206,51.18,48.82
3,60-69,174,101,63.27,36.73
4,70+,14,20,41.18,58.82


## D. Variables restantes y matriz de hallazgos

Se identifican categorías con pocos registros o distribuciones concentradas. La matriz separa evidencia observada, interpretación prudente y riesgo potencial; no diagnostica sesgo ni discriminación.

In [5]:
discrete = [c for c in df.columns if c not in ['target', 'sex'] and df[c].nunique() <= 10]
concentration_rows = []
for column in discrete:
    shares = df[column].value_counts(normalize=True) * 100
    concentration_rows.append({'variable': column, 'categoria_mayoritaria': shares.index[0], 'porcentaje_mayoritario': round(shares.iloc[0], 2), 'categorias_menos_5pct': int((shares < 5).sum())})
concentration = pd.DataFrame(concentration_rows)
display(concentration)
concentration.to_csv(TABLES / 'concentracion_variables_discretas.csv', index=False)
hallazgos = findings_matrix(df, age_group)
display(hallazgos)
hallazgos.to_csv(TABLES / 'hallazgos_eda.csv', index=False)
print('Tablas y gráficos exportados en:', ROOT / 'outputs')

,variable,categoria_mayoritaria,porcentaje_mayoritario,categorias_menos_5pct
0,cp,0,48.49,0
1,fbs,0,85.07,0
2,restecg,1,50.05,1
3,exang,0,66.34,0
4,slope,1,47.02,0
5,ca,0,56.39,1
6,thal,2,53.07,1


,id,dimension,evidencia,valor_observado,posible_problema,tipo_sesgo_potencial,impacto_potencial,requiere_revision
0,S01,sex,El grupo menos representado por sex concentra ...,sex=0: 30.44%,Representación desigual entre grupos.,Posible sesgo de representación.,Un eventual modelo podría generalizar peor par...,Sí
1,S02,age_group,El rango etario presenta una participación inf...,<40: 5.56%,Grupo etario poco representado.,Posible sesgo de representación.,Menor estabilidad descriptiva y posible menor ...,Sí
2,S03,age_group,El rango etario presenta una participación inf...,70+: 3.32%,Grupo etario poco representado.,Posible sesgo de representación.,Menor estabilidad descriptiva y posible menor ...,Sí
3,S04,calidad_datos,Se observan filas duplicadas exactas en el dat...,723 duplicados (70.54%),Sobre-representación de registros repetidos.,Posible riesgo de representatividad.,Podría afectar análisis descriptivos y una eva...,Sí


Tablas y gráficos exportados en: C:\Users\cesar\OneDrive\Desktop\DuocUC\3erYear\GestionDeProyectoDeDatos\eva1\outputs
